# Geno prep CONGRADS
1. Separate binary and quanntitative covs
2. Residualize and quantile transform phenos

In [1]:
import pandas as pd
import os
import statsmodels.api as sm
from sklearn.preprocessing import quantile_transform, RobustScaler
import numpy as np
import glob

workspace_path = "/data/workspaces/lag/workspaces/lg-ukbiobank/projects/CONGRADS_rest/"
cfs_path = "/data/clusterfs/lag/users/jitame/CONGRADS/pheno"

In [2]:
covs = pd.read_csv(os.path.join(workspace_path, "regenie_covariates_65k.tsv"), sep="\t")
print(covs.shape)
covs.dropna(axis=1, how="all", inplace=True)
print(covs.shape)
covs = covs.drop(["site_dummy_11028"], axis=1)
print(covs.shape)
covs.dropna(axis=0, how="any", inplace=True)
print(covs.shape)

binary_covs = ["Genetic_sex", "geno_array_dummy", "site_dummy_11025", "site_dummy_11026", "site_dummy_11027"]
for cov in binary_covs[2:]:
    covs[cov] = covs[cov].astype(int)


covs_in = ["FID", "IID"] + list(covs.columns)
covs_in_binary = ["FID", "IID"] + binary_covs
covs[covs_in].to_csv(os.path.join(cfs_path, "regenie_final_covs.tsv"), sep="\t", header=True, index=False, na_rep="NA")
covs[covs_in_binary].to_csv(os.path.join(cfs_path, "gcta_binary_covs.tsv"), sep="\t", header=False, index=False)

covs.drop(binary_covs, axis=1).to_csv(os.path.join(cfs_path, "gcta_qcovs.tsv"), sep="\t", header=False, index=False)
covs.set_index(["IID"], inplace=True)



(46417, 29)
(46417, 29)
(46417, 28)
(46363, 28)


In [3]:
#data_files = [os.path.join(cfs_path, "congrads_tcs_pcs_1std_N34350.tsv"), os.path.join(cfs_path, "congrads_tcs_pcs_N34545.tsv")]
#data_files = [os.path.join(cfs_path, "congrads_pca_g0g1_N34545.tsv"), os.path.join(cfs_path, "congrads_pca_g0g1_1std_N34350.tsv")]
#data_files = [os.path.join(cfs_path, "congrads_pca_white_g0g1_new_N34545.tsv")]
#data_files = [ os.path.join(cfs_path, "melodic_pcs_g0_N34505.tsv"), os.path.join(cfs_path, "melodic_ics_g0_N34505.tsv") ]
#data_files =  [ os.path.join(cfs_path, "melodic_ics_g0_pmaps_N34444.tsv") ]
#data_files = [os.path.join(cfs_path, "CONGRADS_pmap_lstg_glasser_N33357.tsv") ]
#data_files = [os.path.join(cfs_path, "CONGRADS_pmap_lifg_glasser_N34170.tsv") ]
#data_files = [ os.path.join(cfs_path, "all_idps_congrads.tsv"), os.path.join(workspace_path, "all_idps_congrads.tsv") ]
data_files = [ os.path.join(cfs_path, "all_idps_congrads_new.tsv"), os.path.join(workspace_path, "all_idps_congrads_new.tsv") ]
#data_files = [ os.path.join(cfs_path, "congrads_AICHA_fslcc_pca_g0g1_N49259.tsv"), os.path.join(workspace_path, "congrads_AICHA_fslcc_pca_g0g1_N49259.tsv"), 
#              os.path.join(cfs_path, "congrads_glasser_fslcc_pca_g0g1_N49257.tsv"), os.path.join(workspace_path, "congrads_glasser_fslcc_pca_g0g1_N49257.tsv") ]



In [4]:
print(data_files)

['/data/clusterfs/lag/users/jitame/CONGRADS/pheno/all_idps_congrads_new.tsv', '/data/workspaces/lag/workspaces/lg-ukbiobank/projects/CONGRADS_rest/all_idps_congrads_new.tsv']


In [5]:
def save_df(data, file_name):
    #reorder and save
    initial_cols = data.columns
    data['FID'] = data.index.values.astype(int)
    data['IID'] = data.index.values.astype(int)
    data = data[['FID', 'IID', *initial_cols]]
    data.to_csv(file_name.format(len(data)), na_rep="NA", sep="\t", index=False, header=True)

def residualize(data, covs, fn_out):

    #set up files
    subs = sorted(list(set(data["FID"]) & set(covs["FID"])))
    print("Number of subs in both files: {}".format(len(subs)))
    
    data.set_index(["IID"], inplace=True)
    data = data.loc[subs]
    covs = covs.loc[subs]
    data.drop(["FID"], axis=1, inplace=True)
    covs.drop(["FID"], axis=1, inplace=True)

    print("Residualizing...")
    #define new dataframe
    data_new=pd.DataFrame(index=data.index.values)

        #residualize
    for dep_var in data.columns: 
        na_bool = data[dep_var].isna()
        
        if np.sum(na_bool) == len(data):
            continue
        
        data_in = data.loc[~na_bool, dep_var]
        covs_in = covs.loc[~na_bool, :]
        model = sm.OLS(data_in, exog=covs_in)
        results = model.fit()
        df_residualized = pd.DataFrame(data=results.resid, index=data_in.index, columns=[dep_var])
        data_new = data_new.join(df_residualized)
        print(data_new.shape)
    
    #print("Quantile transform...")
    #print("Scale to IQR...")
    #quantile transformation
    X = data_new.to_numpy()
    data_new2 = pd.DataFrame(data=quantile_transform(X, n_quantiles=1000, output_distribution='normal', random_state=0, copy=True),
                             columns=data_new.columns,
                             index=data_new.index.values)
    #data_new2 = pd.DataFrame(data=RobustScaler(quantile_range=(25, 75), with_centering=False, with_scaling=True).fit_transform(X),
    #                         columns=data_new.columns,
    #                         index=data_new.index.values)
    
    
    
    print("Saving results...")
    save_df(data = data_new,
            file_name=fn_out[:-4]+"_resid_N{}.tsv".format(len(data_new)))
    save_df(data = data_new2,
            file_name=fn_out[:-4]+"_resid_norm_N{}.tsv".format(len(data_new2)))
    print("Done!")      


In [6]:
for fn in data_files:
    print(fn)
    data = pd.read_csv(fn, sep="\t")
    residualize(data, covs, fn)

/data/clusterfs/lag/users/jitame/CONGRADS/pheno/all_idps_congrads_new.tsv
Number of subs in both files: 46267
Residualizing...
(46267, 1)
(46267, 2)
(46267, 3)
(46267, 4)
(46267, 5)
(46267, 6)
(46267, 7)
(46267, 8)
(46267, 9)
(46267, 10)
(46267, 11)
(46267, 12)
(46267, 13)
(46267, 14)
(46267, 15)
(46267, 16)
(46267, 17)
(46267, 18)
(46267, 19)
(46267, 20)
(46267, 21)
(46267, 22)
(46267, 23)
(46267, 24)
(46267, 25)
(46267, 26)
(46267, 27)
(46267, 28)
(46267, 29)
(46267, 30)
(46267, 31)
(46267, 32)
(46267, 33)
(46267, 34)
(46267, 35)
(46267, 36)
(46267, 37)
(46267, 38)
(46267, 39)
(46267, 40)
(46267, 41)
(46267, 42)
(46267, 43)
(46267, 44)
(46267, 45)
(46267, 46)
(46267, 47)
(46267, 48)
(46267, 49)
(46267, 50)
(46267, 51)
(46267, 52)
(46267, 53)
(46267, 54)
(46267, 55)
(46267, 56)
(46267, 57)
(46267, 58)
(46267, 59)
(46267, 60)
(46267, 61)
(46267, 62)
(46267, 63)
(46267, 64)
(46267, 65)
(46267, 66)
(46267, 67)
(46267, 68)
(46267, 69)
(46267, 70)
(46267, 71)
(46267, 72)
(46267, 73)
(46267

In [7]:
#covs = pd.read_csv(os.path.join(workspace_path, "regenie_covariates_65k.tsv"), sep="\t")
subs = sorted(list(set(data["FID"]) & set(covs["FID"])))
print(len(subs))

46267


In [9]:
def to_list(in_list, fn):
    with open(fn, "w") as file:
        for row in in_list:
            file.write(str(row)+'\n')
            
to_list(subs, fn=os.path.join(workspace_path, "congrads_final_subs_N{}.txt".format(len(subs))))
to_list(subs, fn=os.path.join(cfs_path, "congrads_final_subs_N{}.txt".format(len(subs))))